# 05 - Métodos de ponto fixo
Vamos aprender sobre como usar os métodos de ponto fixo para encontrar as raízes de funções reais.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana5
```

Faça o checkout nessa nova branch:

```bash
git checkout semana5
```

Instale as bibliotecas NumPy e SciPy.

In [1]:
%pip install numpy scipy

Dependências já instaladas no ambiente de execução.


<hr />

## Atividade 1
Escreva os métodos de ponto fixo dentro do arquivo $utils/algoritmos.py$.

In [1]:
import numpy as np
from scipy.optimize import approx_fprime

def pontofixo(a, g, TOL=1e-8):
    x = g(a)
    while abs(x - a) > TOL:
        a = x
        x = g(a)
    return x

def newton_raphson(a, f, TOL=1e-8, df=None):
    if df is None:
        def dfn(x):
            return approx_fprime(np.array([x]), lambda v: f(v[0]))[0]
    else:
        dfn = df
    g = lambda x: x - f(x) / dfn(x)
    return pontofixo(a, g, TOL)

def secante(a, b, f, TOL=1e-8):
    g = lambda a, b: (a * f(b) - b * f(a)) / (f(b) - f(a))
    x = g(a, b)
    while abs(x - b) > TOL:
        a, b = b, x
        x = g(a, b)
    return x

Resolver a equação $e^x = x + 2$ é equivalente a calcular os pontos fixos da função  

$$
g(x) = e^x - 2
$$

Use os métodos do ponto fixo  

$$
x^{(n+1)} = g(x^{(n)})
$$

com $x^{(0)} = -1.8$ para obter uma aproximação de uma das soluções da equação dada com 8 dígitos significativos.  

**Resposta:**  
$
x \approx -1.8414057
$

In [1]:
import numpy as np

import sys
import os

# Sobe um nível para a raiz do projeto e adiciona ao caminho do sistema
sys.path.append(os.path.abspath(os.path.join('..')))

from utils.algoritmos import (
    pontofixo,
    newton_raphson,
    secante,
)

f1 = lambda x: np.e**x - x - 2
g1 = lambda x: np.e**x - 2


def main():
    r = pontofixo(-1.8, g1)
    print(f"raiz ponto fixo = {r}")
    r = newton_raphson(-1.8, f1, df=lambda x: np.e**x - 1)
    print(f"raiz newton-raphson df = {r}")
    r = newton_raphson(-1.8, f1)
    print(f"raiz newton-raphson = {r}")
    r = secante(-1.8, -1.7, f1)
    print(f"raiz secante = {r}")


if __name__ == "__main__":
    main()

raiz ponto fixo = -1.841405660009702
raiz newton-raphson df = -1.8414056604369606
raiz newton-raphson = -1.8414056604369609
raiz secante = -1.84140565996016


## Atividade 2
Encontre a raiz positiva da função  

$$
f(x) = \cos(x) - x^2
$$  

pelos métodos do ponto fixo, inicializando-o com $x^{(0)} = 1$.  

Realize a iteração até obter estabilidade no **quinto dígito significativo**.  

**Resposta:**

$$
x \approx 0.82413 
$$

Processo iterativo:  

$$
x^{(n+1)} = x^{(n)} + \frac{\cos(x) - x^2}{\sin(x) + 2x}
$$

In [1]:
f2 = lambda x: np.cos(x) - x*x
g2 = lambda x: x + (np.cos(x)-x*x)/(np.sin(x)+2*x)
df2 = lambda x: -np.sin(x) - 2*x

fixed_root = pontofixo(1.0, g2, TOL=5e-6)
newton_root = newton_raphson(1.0, f2, TOL=5e-6, df=df2)
secant_root = secante(0.8, 1.0, f2, TOL=5e-6)
print(f"ponto fixo: {fixed_root:.5f}")
print(f"Newton-Raphson: {newton_root:.5f}")
print(f"secante: {secant_root:.5f}")


ponto fixo: 0.82413
Newton-Raphson: 0.82413
secante: 0.82413


## Atividade 3
Aplique os métodos do ponto fixo para resolver a equação:

$$
e^{-x^2} = 2x
$$

**Resposta:**

$$
x \approx 0.4193648
$$

In [1]:
f3 = lambda x: np.exp(-x*x) - 2*x
g3 = lambda x: 0.5*np.exp(-x*x)
df3 = lambda x: -2*x*np.exp(-x*x) - 2

print(f"ponto fixo: {pontofixo(0.5, g3):.8f}")
print(f"Newton-Raphson: {newton_raphson(0.5, f3, df=df3):.8f}")
print(f"secante: {secante(0.4, 0.5, f3):.8f}")


ponto fixo: 0.41936482
Newton-Raphson: 0.41936482
secante: 0.41936482


## Atividade 4
Resolva os três últimos exercícios da semana anterior usando os métodos do ponto fixo.

In [1]:
# Exercícios 4, 5 e 6 da semana anterior resolvidos por métodos abertos.
def diode_equation(vd, voltage, resistance):
    reverse_current = 1e-12
    thermal_voltage = 1.38064852e-23*300/1.60217662e-19
    return resistance*reverse_current*np.expm1(vd/thermal_voltage) + vd - voltage


def diode_derivative(vd, resistance):
    thermal_voltage = 1.38064852e-23*300/1.60217662e-19
    return resistance*1e-12*np.exp(vd/thermal_voltage)/thermal_voltage + 1


print("Circuito com diodo — Newton-Raphson")
for voltage, resistance, guess in [(30,1e3,.6),(3,1e3,.55),(3,1e4,.5),(.3,1e3,.3),(-.3,1e3,-.3),(-30,1e3,-30),(-30,1e4,-30)]:
    function = lambda x, v=voltage, r=resistance: diode_equation(x, v, r)
    derivative = lambda x, r=resistance: diode_derivative(x, r)
    root = newton_raphson(guess, function, df=derivative)
    print(f"V={voltage:g} V, R={resistance/1e3:g} kΩ: vd={root:.3f} V")


def catenary(c):
    return c*(np.cosh(250/c)-1)-50


def catenary_derivative(c):
    ratio = 250/c
    return np.cosh(ratio)-ratio*np.sinh(ratio)-1


print(f"Catenária — Newton: C={newton_raphson(600, catenary, df=catenary_derivative):.4f} m")
print(f"Catenária — secante: C={secante(550, 700, catenary):.4f} m")


tan_phi = 2*np.pi*1e3*100e-3/1e3
phi = np.arctan(tan_phi)
rectifier = lambda beta: np.sin(beta-phi)+np.sin(phi)*np.exp(-beta/tan_phi)
rectifier_derivative = lambda beta: np.cos(beta-phi)-np.sin(phi)*np.exp(-beta/tan_phi)/tan_phi
newton_beta = newton_raphson(np.radians(212.5), rectifier, df=rectifier_derivative)
secant_beta = secante(np.radians(212), np.radians(213), rectifier)
print(f"Retificador — Newton: beta={np.degrees(newton_beta):.4f}°")
print(f"Retificador — secante: beta={np.degrees(secant_beta):.4f}°")


Circuito com diodo — Newton-Raphson
V=30 V, R=1 kΩ: vd=0.623 V
V=3 V, R=1 kΩ: vd=0.559 V
V=3 V, R=10 kΩ: vd=0.500 V
V=0.3 V, R=1 kΩ: vd=0.300 V
V=-0.3 V, R=1 kΩ: vd=-0.300 V
V=-30 V, R=1 kΩ: vd=-30.000 V
V=-30 V, R=10 kΩ: vd=-30.000 V
Catenária — Newton: C=633.1622 m
Catenária — secante: C=633.1622 m
Retificador — Newton: beta=212.2258°
Retificador — secante: beta=212.2258°


## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 5"
git push origin semana5
```